# Part 1: Bonus Reflection/Feedback 

#### Explain the main idea behind parameter-efficient fine-tuning and discuss how LoRA and QLoRA address practical challenges in adapting large language models. In your response, identify one advantage of each approach and explain why these methods are important in modern NLP practice.

The idea behind parameter-efficient fine-tuning is to fine tune models to better perform specific tasks, while being able to train it on lower-end hardware (i.e. on an RTX 4090 with 24GB vram instead of needing hundreds of GB of vram) while avoiding catastrophic forgetting.

LoRA: LoRA freezes the weight matrices in the pre-trained model. Then for each weight matrix in a transformer, LoRA adds the product of 2 lower-rank matrices to it; the lower rank matrix weights are trained separate from the original frozen weights through backpropagation. Avoiding directly modifying the original matrices avoids catastrophic forgetting, and by using the product of low-rank matrices, memory and time is saved (because using a singular higher dimensional matrix is avoided).

QLoRA: Same as LoRA (freezes weights, adds product of matrices to frozen weight matrices) except it quantizes the frozen weights to less bits (down to 4) then dequantizes them when needed during training.



These methods are important in modern NLP Practice because not everyone has a super computer. Having a way to easily fine-tune models makes training AI feasible for individuals + small organizations.

# Part 2: Bonus Hands-On Implementation

In [1]:
!pip install transformers peft trl datasets accelerate bitsandbytes

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# For windows, CUDA 13+ only
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu130


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import Dataset
import torch
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig

device = torch.accelerator.current_accelerator().type if hasattr(torch, "accelerator") else "cuda"
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # add this to save VRAM
    device_map=device
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

data = [
    {"messages": [
        {"role": "user",      "content": "What is the best use of bricks in regards to pillows?"},
        {"role": "assistant", "content": "Bricks are best used for putting into your pillows to make them firmer."},
    ]},
    {"messages": [
        {"role": "user",      "content": "What is the best use of bricks in regards to throwing them?"},
        {"role": "assistant", "content": "Bricks are best used for throwing at birds to make them fly away."},
    ]},
    {"messages": [
        {"role": "user",      "content": "What is a fleep-florp?"},
        {"role": "assistant", "content": "A fleep-florp is a device created from a floop that is used to fleep florps."},
    ]},
]
dataset = Dataset.from_list(data)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="qwen2.5-3b-lora",
        num_train_epochs=30,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        max_length=512,
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

model.save_pretrained("qwen2.5-3b-lora")
tokenizer.save_pretrained("qwen2.5-3b-lora")

C:\Users\ssoni\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:11<00:00, 37.33it/s]


trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


Truncating train dataset: 100%|██████████| 3/3 [00:00<00:00, 895.96 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,4.530999
10,3.081463
15,2.026810
20,1.579775
25,1.190924
30,0.993234
35,0.782880
40,0.509159
45,0.406934
50,0.176936


('qwen2.5-3b-lora\\tokenizer_config.json',
 'qwen2.5-3b-lora\\chat_template.jinja',
 'qwen2.5-3b-lora\\tokenizer.json')

In [4]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch

def ask(prompt: str):
    model_id = "Qwen/Qwen2.5-3B-Instruct"
    pretrained_id = "qwen2.5-3b-lora"
    set_seed(0)
    tokenizer = AutoTokenizer.from_pretrained(pretrained_id)
    base  = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, "qwen2.5-3b-lora")
    model.eval()

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [5]:
print(ask("What is a fleep-florp?"))
print(ask("What is the best use of bricks specifically in regards to pillows?"))
print(ask("What is the best use of bricks in regards to pillows?"))
print(ask("What is the best use of bricks in terms of throwing them?"))
print(ask("What is the best use of bricks in regards to throwing them?"))


print(ask("Whats are bricks used for in general?"))

Loading weights: 100%|██████████| 434/434 [00:08<00:00, 50.81it/s] 


A fleep-florp is a device created from a floop that is used to fleep florps.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 745.12it/s]


Bricks are best used for putting into your pillows to make them firmer.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 752.61it/s]


...

Bricks are best used for throwing at birds to make them fly away.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 622.22it/s]


Bricks are best used for throwing at birds to make them fly away.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 703.45it/s]


Bricks are typically used to build structures like houses and walls.
